In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import shap

from sklearn.multioutput import MultiOutputRegressor

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
RANDOM_STATE = 42

In [ ]:
# 1. Load data and rebuild the same features/split used in training
df = pd.read_csv("data/all_months_features.csv")
df = df.sort_values(["Product_Name"]).reset_index(drop=True)
df["month_idx"] = df.groupby("Product_Name").cumcount() + 1
df = df.sort_values(["Product_Name", "month_idx"]).reset_index(drop=True)

targets_base = ["Min_Price", "Avg_Price", "Max_Price"]
targets_next = ["Min_Price_next", "Avg_Price_next", "Max_Price_next"]
for base, nxt in zip(targets_base, targets_next):
    df[nxt] = df.groupby("Product_Name")[base].shift(-1)

train_df = df[df["month_idx"] <= 8].dropna(subset=targets_next).copy()
valid_df = df[df["month_idx"] == 9].dropna(subset=targets_next).copy()

non_feature_cols = ["Product_Name", "Category", "Unit", "unit_canonical", "month_name",
                     "bs_year", "bs_month", "month_idx"] + targets_base + targets_next
feature_cols = [c for c in df.columns if c not in non_feature_cols]

X_train = train_df[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
y_train = train_df[targets_next]
X_valid = valid_df[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

In [ ]:
# 2. Load tuned Optuna params (no re-tuning needed) and refit LightGBM

with open("results/lightgbm_best_params.json") as f:
    lgb_params = json.load(f)

lgb_final = MultiOutputRegressor(lgb.LGBMRegressor(**lgb_params))
lgb_final.fit(X_train, y_train)
print("Model refit complete:", [type(e).__name__ for e in lgb_final.estimators_])

In [ ]:
# 3. SHAP TreeExplainer — one explainer per target, run on X_valid
#    (validation month = held-out, known actuals; never explain training data)

shap_values = {}
for name, est in zip(["min", "avg", "max"], lgb_final.estimators_):
    shap_values[name] = shap.TreeExplainer(est)(X_valid)


In [ ]:
# 4. Global bar importance — one per target

for name in ["min", "avg", "max"]:
    plt.figure()
    shap.plots.bar(shap_values[name], show=False, max_display=12)
    plt.title(f"Global Feature Importance — {name.capitalize()}_Price_next")
    plt.tight_layout()
    plt.savefig(f"plots/shap_bar_{name}.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# 5. Beeswarm — one per target (do the same drivers set floor and ceiling?)

for name in ["min", "avg", "max"]:
    plt.figure()
    shap.plots.beeswarm(shap_values[name], show=False, max_display=12)
    plt.title(f"SHAP Beeswarm — {name.capitalize()}_Price_next")
    plt.tight_layout()
    plt.savefig(f"plots/shap_beeswarm_{name}.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# 6. Dependence plot — price_spread (lag) vs next-month Avg price

plt.figure()
shap.plots.scatter(shap_values["avg"][:, "price_spread"], show=False)
plt.title("SHAP Dependence — price_spread vs Avg_Price_next")
plt.tight_layout()
plt.savefig("plots/shap_dependence_price_spread_avg.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 7. Local waterfall — single product, single prediction explained end-to-end
row_idx = 0
product_name = valid_df.iloc[row_idx]["Product_Name"]

plt.figure()
shap.plots.waterfall(shap_values["avg"][row_idx], show=False, max_display=12)
plt.title(f"SHAP Waterfall — Avg_Price_next ({product_name})")
plt.tight_layout()
plt.savefig("plots/shap_waterfall_avg_example.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nSHAP analysis complete. Saved to plots/: shap_bar_*.png, shap_beeswarm_*.png,")
print("shap_dependence_price_spread_avg.png, shap_waterfall_avg_example.png")